# Week 04A — Classical Segmentation: Thresholds, Edges and the Hough Transform

**MCTA 4364 Machine Vision** · Session 1 of Week 4 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W04_segmentation/W04A_thresholds_edges_hough.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W04_segmentation/W04A_thresholds_edges_hough.ipynb)

**Segmentation** partitions an image into meaningful regions: part vs background, defect vs good surface, each coin separately. It is the
bridge from pixels to objects that can be counted, measured and classified. This session covers the two classic strategies:
**similarity** (group pixels that look alike, i.e. *thresholding*) and **discontinuity** (find where pixels change, i.e. *edges*), plus the
**Hough transform** for finding lines and circles in edge maps.

### Learning outcomes
By the end of this session you will be able to:
1. Choose a threshold from a histogram and derive **Otsu's method** (maximise between-class variance).
2. Apply **adaptive** thresholding for uneven lighting, and **multi-level** thresholds for more than two classes.
3. **Evaluate** a segmentation against ground truth with IoU, Dice, precision and recall.
4. Explain the four stages of the **Canny** edge detector and tune its thresholds.
5. Detect **lines and circles** with the Hough transform, and understand its voting accumulator.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–25 | 1–3 | Global, Otsu, multi-level and adaptive thresholding (interactive) |
| 25–35 | 4 | Measuring segmentation quality |
| 35–55 | 5 | Canny edge detection, stage by stage |
| 55–72 | 6 | Hough lines and circles |
| 72–80 | 9 | Quiz and exit ticket |
| Home | 7–8 | Exercises with self-checks, challenge |

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import skimage.data
from skimage.filters import threshold_multiotsu, threshold_sauvola

from cvhelpers import check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
rng = np.random.default_rng(0)
coins = skimage.data.coins()
cam = skimage.data.camera()
sudoku = load_sample("sudoku.png", gray=True)
building = load_sample("building.jpg")

## 1. Global thresholding and Otsu's method

The simplest segmentation: $\text{mask}(x, y) = [\,f(x, y) > T\,]$. It works when the histogram is **bimodal**: one peak for the background, one
for the objects, with a valley between them. The question is how to choose $T$ automatically.

**Otsu's method (1979)** tries every $T$ and picks the one that best separates the two classes. For a threshold $t$, let $\omega_0, \omega_1$ be the
fractions of pixels below and above $t$ and $\mu_0, \mu_1$ their mean intensities. Otsu maximises the **between-class variance**

$$ \sigma_B^2(t) = \omega_0(t)\,\omega_1(t)\,\big[\mu_0(t) - \mu_1(t)\big]^2 $$

which is equivalent to minimising the variance *within* each class. Everything can be computed from cumulative sums of the histogram,
so the search over all 256 thresholds is very fast.

In [ ]:
hist = np.bincount(coins.ravel(), minlength=256).astype(float)
p = hist / hist.sum()
levels = np.arange(256)
w0 = np.cumsum(p); w1 = 1 - w0
mu_cum = np.cumsum(p * levels); mu_total = mu_cum[-1]
with np.errstate(divide="ignore", invalid="ignore"):
    mu0 = mu_cum / w0
    mu1 = (mu_total - mu_cum) / w1
    sigma_b = np.nan_to_num(w0 * w1 * (mu0 - mu1) ** 2)
t_otsu = int(sigma_b.argmax())
t_cv, mask_otsu = cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={"width_ratios": [1.4, 1, 1]})
axes[0].bar(levels, hist, width=1, color="#9aa0a6")
ax2 = axes[0].twinx(); ax2.plot(levels, sigma_b, color="#d93025", lw=2.5); ax2.set_ylabel("between-class variance", color="#d93025")
axes[0].axvline(t_otsu, color="#188038", lw=2, ls="--"); axes[0].text(t_otsu + 3, hist.max() * 0.9, f"Otsu t = {t_otsu}", color="#188038")
axes[0].set_xlabel("grey level"); axes[0].set_title("histogram (grey) and Otsu criterion (red)")
axes[1].imshow(coins, cmap="gray"); axes[1].set_title("coins"); axes[1].axis("off")
axes[2].imshow(mask_otsu, cmap="gray"); axes[2].set_title(f"Otsu mask (OpenCV t = {t_cv:.0f})"); axes[2].axis("off")
plt.suptitle("Figure 1.1 - Otsu picks the threshold that maximises the separation of the two classes", fontweight="bold")
plt.tight_layout(); plt.show()

**Other automatic thresholds.** Otsu assumes two classes of reasonable size. When the objects are *small* (a few scratches on a large surface),
the **triangle** method, which finds the point of the histogram farthest from a line joining its peak to its tail, often works better. For **three or
more** classes (background, part, shadow), **multi-Otsu** generalises the criterion to several thresholds.

In [ ]:
t_tri, mask_tri = cv2.threshold(coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_TRIANGLE)
multi = threshold_multiotsu(cam, classes=3)
regions = np.digitize(cam, bins=multi)
fig, axes = plt.subplots(1, 4, figsize=(21, 4.6))
axes[0].imshow(mask_tri, cmap="gray"); axes[0].set_title(f"triangle method on coins (t = {t_tri:.0f})")
axes[1].hist(cam.ravel(), 256, range=(0, 256), color="#9aa0a6")
for t in multi:
    axes[1].axvline(t, color="#d93025", lw=2)
axes[1].set_title(f"camera histogram, multi-Otsu thresholds {multi.astype(int)}")
axes[2].imshow(cam, cmap="gray"); axes[2].set_title("camera")
axes[3].imshow(regions, cmap="viridis"); axes[3].set_title("3 classes: dark coat, grass, sky")
for ax in (axes[0], axes[2], axes[3]):
    ax.axis("off")
plt.suptitle("Figure 1.2 - Triangle and multi-level Otsu thresholds", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Adaptive (local) thresholding

With **uneven illumination** no single threshold works everywhere (Week 3A). **Adaptive** methods compute a separate threshold for every pixel from
its neighbourhood:
- **mean / Gaussian**: $T(x, y) = \text{mean}_{w \times w}(x, y) - C$ (OpenCV `adaptiveThreshold`),
- **Sauvola** (document binarisation): $T = m\,\big[1 + k\,(s/R - 1)\big]$, which also adapts to the local standard deviation $s$.

The window must be **larger than the objects** (strokes, characters) but **smaller than the lighting variation**.

In [ ]:
_, glob = cv2.threshold(sudoku, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
ad_mean = cv2.adaptiveThreshold(sudoku, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 25, 10)
ad_gauss = cv2.adaptiveThreshold(sudoku, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 25, 10)
sauvola = (sudoku > threshold_sauvola(sudoku, window_size=25)).astype(np.uint8) * 255
show(sudoku, glob, ad_mean, ad_gauss, sauvola, titles=["uneven lighting", "global Otsu", "adaptive mean (25, C=10)",
                                                      "adaptive Gaussian", "Sauvola"], figsize=(24, 5))

### Interactive: threshold explorer

In [ ]:
def threshold_demo(method="adaptive mean", t=128, block=25, C=10, image="sudoku"):
    src = {"sudoku": sudoku, "coins": coins, "camera": cam}[image]
    b = block if block % 2 else block + 1
    if method == "global":
        out = ((src > t) * 255).astype(np.uint8)
    elif method == "Otsu":
        t, out = cv2.threshold(src, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        mode = cv2.ADAPTIVE_THRESH_MEAN_C if method == "adaptive mean" else cv2.ADAPTIVE_THRESH_GAUSSIAN_C
        out = cv2.adaptiveThreshold(src, 255, mode, cv2.THRESH_BINARY, b, C)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.2]})
    axes[0].imshow(src, cmap="gray"); axes[1].imshow(out, cmap="gray")
    axes[1].set_title(f"{method}" + (f", t = {t:.0f}" if method in ("global", "Otsu") else f", block {b}, C {C}"))
    axes[2].hist(src.ravel(), 256, range=(0, 256), color="#9aa0a6")
    if method in ("global", "Otsu"):
        axes[2].axvline(t, color="#d93025", lw=2)
    for ax in axes[:2]:
        ax.axis("off")
    plt.tight_layout(); plt.show()


interact(threshold_demo, method=["adaptive mean", "adaptive Gaussian", "global", "Otsu"], t=widgets.IntSlider(value=128, min=0, max=255),
         block=widgets.IntSlider(value=25, min=3, max=151, step=2), C=widgets.IntSlider(value=10, min=-20, max=40), image=["sudoku", "coins", "camera"])

## 3. Measuring segmentation quality

To compare methods objectively we need **ground truth** (a correct mask). With predicted mask $P$ and ground truth $G$:

$$ \text{IoU} = \frac{|P \cap G|}{|P \cup G|}, \qquad \text{Dice} = \frac{2|P \cap G|}{|P| + |G|}, \qquad
\text{precision} = \frac{|P \cap G|}{|P|}, \qquad \text{recall} = \frac{|P \cap G|}{|G|} $$

IoU (Jaccard index) and Dice range from 0 (no overlap) to 1 (perfect). Precision asks *how much of what I found is real*; recall asks *how much of
what is real did I find*. Industrial tasks often weigh them differently: missing a defect (low recall) may be far worse than a false alarm.

We create synthetic parts with **known ground truth**, add realistic problems (an illumination gradient, noise, a shadow) and benchmark every method.

In [ ]:
def synthetic_parts(seed, gradient=0.6, noise=12):
    r = np.random.default_rng(seed)
    gt = np.zeros((300, 400), np.uint8)
    for _ in range(7):
        c = (int(r.integers(40, 360)), int(r.integers(40, 260)))
        if r.random() < 0.5:
            cv2.circle(gt, c, int(r.integers(15, 35)), 1, -1)
        else:
            cv2.rectangle(gt, (c[0] - 25, c[1] - 15), (c[0] + 25, c[1] + 15), 1, -1)
    xx = np.linspace(0, 1, 400)[None, :].repeat(300, 0)
    light = 1 - gradient * xx
    img = (60 + 110 * gt) * light + r.normal(0, noise, gt.shape)
    return np.clip(img, 0, 255).astype(np.uint8), gt.astype(bool)


def scores(pred, gt):
    pred = pred > 0
    inter, union = (pred & gt).sum(), (pred | gt).sum()
    return {"IoU": inter / union, "Dice": 2 * inter / (pred.sum() + gt.sum()),
            "precision": inter / max(pred.sum(), 1), "recall": inter / gt.sum()}


def flat_field_otsu(im):
    """Divide by an estimate of the illumination, then Otsu.

    The background is estimated with a large morphological opening. An opening is a MINIMUM filter followed by a
    maximum filter, so on a noisy image it latches onto noise minima: denoise with a median filter first!
    """
    background = cv2.morphologyEx(cv2.medianBlur(im, 5), cv2.MORPH_OPEN, np.ones((81, 81), np.uint8))
    background = cv2.GaussianBlur(background, (0, 0), 20).astype(np.float32)
    ratio = np.clip(cv2.GaussianBlur(im, (0, 0), 2) / (background + 1) * 100, 0, 255).astype(np.uint8)
    return cv2.threshold(ratio, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]


methods = {
    "global t=110": lambda im: im > 110,
    "Otsu": lambda im: cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1],
    "adaptive mean": lambda im: cv2.adaptiveThreshold(cv2.GaussianBlur(im, (5, 5), 0), 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 101, -15),
    "flat-field + Otsu": lambda im: flat_field_otsu(im),
}
img_s, gt_s = synthetic_parts(1)
fig, axes = plt.subplots(1, 2 + len(methods), figsize=(24, 3.8))
axes[0].imshow(img_s, cmap="gray"); axes[0].set_title("synthetic parts, light falls to the right")
axes[1].imshow(gt_s, cmap="gray"); axes[1].set_title("ground truth")
for ax, (name, fn) in zip(axes[2:], methods.items()):
    pred = fn(img_s) > 0
    err = np.dstack([pred & ~gt_s, pred & gt_s, ~pred & gt_s]).astype(float)   # red FP, green TP, blue FN
    ax.imshow(err); ax.set_title(f"{name}: IoU {scores(pred, gt_s)['IoU']:.2f}")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 3.1 - Errors per method: green = correct, red = false positive, blue = missed", fontweight="bold")
plt.tight_layout(); plt.show()

table = {name: np.mean([[scores(fn(im) > 0, gt)[k] for k in ("IoU", "precision", "recall")]
                        for im, gt in (synthetic_parts(s) for s in range(10))], axis=0) for name, fn in methods.items()}
x = np.arange(len(table))
fig, ax = plt.subplots(figsize=(12, 4))
for k, (metric, col) in enumerate(zip(("IoU", "precision", "recall"), ("#1a73e8", "#188038", "#e37400"))):
    ax.bar(x + (k - 1) * 0.27, [v[k] for v in table.values()], 0.27, label=metric, color=col)
ax.set_xticks(x, list(table)); ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Figure 3.2 - Benchmark on 10 random synthetic images: correct the lighting, or threshold locally", fontweight="bold")
plt.show()

## 4. Edge-based segmentation: the Canny detector

Canny (1986) designed an edge detector for three goals: find all real edges, locate them precisely, and mark each edge only once.
It has four stages:

1. **Smooth** with a Gaussian ($\sigma$) to suppress noise (Week 3B: derivatives amplify noise).
2. **Gradient**: magnitude $|\nabla f|$ and direction $\theta$ with Sobel.
3. **Non-maximum suppression (NMS)**: keep a pixel only if its magnitude is a local maximum *along the gradient direction*, which thins edges to 1 pixel.
4. **Double threshold + hysteresis**: pixels above $T_\text{high}$ are *strong* edges; pixels between $T_\text{low}$ and $T_\text{high}$ are *weak* and are
   kept only if connected to a strong edge. This keeps faint parts of real contours while rejecting isolated noise.

A common heuristic sets the thresholds from the median intensity $m$: $T_\text{low} = (1 - 0.33)\,m$, $T_\text{high} = (1 + 0.33)\,m$.

In [ ]:
def canny_stages(img, sigma=1.4, low=40, high=90):
    blur = cv2.GaussianBlur(img, (0, 0), sigma).astype(np.float32)
    gx, gy = cv2.Sobel(blur, cv2.CV_32F, 1, 0), cv2.Sobel(blur, cv2.CV_32F, 0, 1)
    mag = np.hypot(gx, gy)
    ang = (np.rad2deg(np.arctan2(gy, gx)) + 180) % 180
    q = (np.round(ang / 45) % 4).astype(int)          # 0: horizontal gradient, 1: 45 deg, 2: vertical, 3: 135 deg
    offsets = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}
    nms = np.zeros_like(mag)
    for k, (dy, dx) in offsets.items():
        fwd = np.roll(mag, (-dy, -dx), axis=(0, 1)); bwd = np.roll(mag, (dy, dx), axis=(0, 1))
        keep = (q == k) & (mag >= fwd) & (mag >= bwd)
        nms[keep] = mag[keep]
    strong, weak = nms >= high, (nms >= low) & (nms < high)
    n, labels = cv2.connectedComponents((strong | weak).astype(np.uint8), connectivity=8)
    keep_ids = np.unique(labels[strong])
    final = np.isin(labels, keep_ids[keep_ids > 0])
    return blur, mag, nms, strong, weak, final


blur, mag, nms, strong, weak, final = canny_stages(cam)
dual = np.zeros(cam.shape + (3,)); dual[strong] = (1, 1, 1); dual[weak] = (0.2, 0.6, 1.0)
fig, axes = plt.subplots(1, 6, figsize=(26, 4.6))
for ax, im, title in zip(axes, [blur, mag, nms > 0, dual, final, cv2.Canny(cv2.GaussianBlur(cam, (0, 0), 1.4), 40, 90)],
                         ["1. smoothed", "2. gradient magnitude", "3. after NMS (thin)", "4. strong (white) / weak (blue)",
                          "5. hysteresis: weak kept if linked", "OpenCV cv2.Canny"]):
    ax.imshow(im[60:330, 150:420] if im.ndim == 2 else im[60:330, 150:420], cmap="gray"); ax.set_title(title); ax.axis("off")
plt.suptitle("Figure 4.1 - The Canny pipeline, stage by stage (zoomed)", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: Canny parameters
Raise $T_\text{high}$ until only real edges start; then lower $T_\text{low}$ to complete their contours. Try the auto thresholds.

In [ ]:
def canny_demo(sigma=1.4, t_low=40, t_high=90, auto=False, image="camera"):
    src = {"camera": cam, "coins": coins, "building": cv2.cvtColor(building, cv2.COLOR_BGR2GRAY), "sudoku": sudoku}[image]
    blur_ = cv2.GaussianBlur(src, (0, 0), sigma)
    if auto:
        m = np.median(blur_); t_low, t_high = int(max(0, 0.67 * m)), int(min(255, 1.33 * m))
    edges = cv2.Canny(blur_, t_low, t_high)
    show(src, edges, titles=[image, f"Canny sigma={sigma}, T_low={t_low}, T_high={t_high}{' (auto)' if auto else ''}"], figsize=(13, 5))


interact(canny_demo, sigma=widgets.FloatSlider(value=1.4, min=0.5, max=5, step=0.1), t_low=widgets.IntSlider(value=40, min=0, max=255),
         t_high=widgets.IntSlider(value=90, min=0, max=255), auto=False, image=["camera", "coins", "building", "sudoku"])

## 5. The Hough transform: finding lines and circles by voting

Edge pixels are scattered; the Hough transform groups them into **parametric shapes**. A line in normal form is

$$ \rho = x\cos\theta + y\sin\theta $$

Each edge point $(x, y)$ is consistent with every line through it, which traces a **sinusoid** in $(\theta, \rho)$ space. Each point **votes** for all bins of an
**accumulator** along its sinusoid. Collinear points' sinusoids intersect in one bin, which collects many votes: a **peak = a line**. Voting is robust to
gaps and noise, and several lines are found at once. Circles work the same way with three parameters $(a, b, r)$; OpenCV's `HoughCircles` uses the
gradient direction to vote efficiently.

In [ ]:
pts = np.array([[20, 80], [40, 70], [60, 60], [80, 50], [30, 20]])
thetas = np.deg2rad(np.arange(0, 180))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
colors = ["#1a73e8"] * 4 + ["#d93025"]
axes[0].scatter(pts[:, 0], pts[:, 1], c=colors, s=80, zorder=3)
xs = np.linspace(0, 100, 10); axes[0].plot(xs, 90 - xs / 2, "--", color="#1a73e8")
axes[0].set_xlim(0, 100); axes[0].set_ylim(100, 0); axes[0].set_aspect("equal"); axes[0].set_title("image space: 4 collinear points + 1 outlier")
for (x, y), c in zip(pts, colors):
    axes[1].plot(np.rad2deg(thetas), x * np.cos(thetas) + y * np.sin(thetas), color=c, lw=2)
t_line = np.arctan2(1, 0.5); r_line = 90 * np.sin(t_line)
axes[1].plot(np.rad2deg(t_line), r_line, "o", ms=16, mfc="none", mec="k", mew=2)
axes[1].set_xlabel("theta (deg)"); axes[1].set_ylabel("rho"); axes[1].grid(alpha=0.3)
axes[1].set_title("Hough space: each point -> a sinusoid; the 4 blue curves meet at the line's (theta, rho)")
plt.suptitle("Figure 5.1 - Point-line duality: collinear points vote for the same accumulator bin", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
canvas = np.zeros((300, 300), np.uint8)
cv2.line(canvas, (20, 250), (280, 60), 255, 2); cv2.line(canvas, (40, 20), (200, 290), 255, 2); cv2.line(canvas, (0, 150), (300, 150), 255, 2)
canvas[rng.random(canvas.shape) < 0.01] = 255                       # noise
ys, xs = np.nonzero(canvas)
diag = int(np.hypot(*canvas.shape))
acc = np.zeros((2 * diag, 180), np.int32)
for t_idx, t in enumerate(thetas):
    rho = np.round(xs * np.cos(t) + ys * np.sin(t)).astype(int) + diag
    np.add.at(acc[:, t_idx], rho, 1)
lines = cv2.HoughLines(canvas, 1, np.pi / 180, 250)
vis = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
for rho, th in lines.reshape(-1, 2):            # (N, 1, 2) in OpenCV 4, (N, 2) in OpenCV 5
    a, b = np.cos(th), np.sin(th)
    x0, y0 = a * rho, b * rho
    cv2.line(vis, (int(x0 + 500 * -b), int(y0 + 500 * a)), (int(x0 - 500 * -b), int(y0 - 500 * a)), (0, 0, 255), 1)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.2), gridspec_kw={"width_ratios": [1, 1.6, 1]})
axes[0].imshow(canvas, cmap="gray"); axes[0].set_title("3 lines + 1% noise pixels")
axes[1].imshow(np.sqrt(acc), cmap="magma", aspect="auto", extent=[0, 180, diag, -diag]); axes[1].set_xlabel("theta (deg)")
axes[1].set_ylabel("rho"); axes[1].set_title("accumulator (sqrt of votes): 3 bright peaks")
axes[2].imshow(vis[..., ::-1]); axes[2].set_title(f"cv2.HoughLines (>= 250 votes): {len(lines)} lines\n(a 2-px thick line can give 2 neighbouring peaks)")
for ax in (axes[0], axes[2]):
    ax.axis("off")
plt.suptitle("Figure 5.2 - The Hough accumulator in action", fontweight="bold")
plt.tight_layout(); plt.show()

**Real images.** `HoughLinesP` (probabilistic) returns line *segments* with end points, which is what most applications need, e.g. the grid of the sudoku or the edges of a building.

In [ ]:
def hough_demo(threshold=80, min_length=60, max_gap=10, image="sudoku"):
    src = {"sudoku": sudoku, "building": cv2.cvtColor(building, cv2.COLOR_BGR2GRAY)}[image]
    edges = cv2.Canny(cv2.GaussianBlur(src, (5, 5), 0), 50, 150)
    segs = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold, minLineLength=min_length, maxLineGap=max_gap)
    vis = cv2.cvtColor(src, cv2.COLOR_GRAY2BGR)
    n = 0 if segs is None else len(segs)
    for x1, y1, x2, y2 in ([] if segs is None else segs.reshape(-1, 4)):
        cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
    show(edges, vis, titles=["Canny edges", f"HoughLinesP: {n} segments"], figsize=(14, 6))


interact(hough_demo, threshold=widgets.IntSlider(value=80, min=10, max=300, step=5), min_length=widgets.IntSlider(value=60, min=5, max=300, step=5),
         max_gap=widgets.IntSlider(value=10, min=0, max=50), image=["sudoku", "building"])

### Circles: sizing coins
Hough circles find the coins directly from the image (no threshold needed). Their radii cluster into sizes, which is the basis of a coin sorter.

In [ ]:
circles = cv2.HoughCircles(cv2.medianBlur(coins, 5), cv2.HOUGH_GRADIENT, dp=1.2, minDist=25, param1=120, param2=32, minRadius=14, maxRadius=40)
vis = cv2.cvtColor(coins, cv2.COLOR_GRAY2BGR)
radii = []
for x, y, r in np.round(circles.reshape(-1, 3)).astype(int):
    radii.append(r)
    cv2.circle(vis, (x, y), r, (0, 200, 0), 2); cv2.circle(vis, (x, y), 2, (0, 0, 255), 3)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.5, 1]})
axes[0].imshow(vis[..., ::-1]); axes[0].axis("off"); axes[0].set_title(f"HoughCircles: {len(radii)} coins")
axes[1].hist(radii, bins=range(12, 42), color="#1a73e8"); axes[1].set_xlabel("radius (px)"); axes[1].set_ylabel("coins")
axes[1].set_title("radius histogram: coin sizes form clusters")
plt.suptitle("Figure 5.3 - Circle detection and sizing", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Otsu from scratch
Return Otsu's threshold for a `uint8` greyscale image using the formula in Section 1 (NumPy only). Pixels **greater than** the threshold are foreground.

In [ ]:
def otsu_threshold(gray_img):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    for im in (coins, cam, sudoku, synthetic_parts(3)[0]):
        t = otsu_threshold(im)
        ref, _ = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        assert abs(int(t) - int(ref)) <= 1, f"expected about {ref:.0f}, got {t}"


check("E1 otsu_threshold", _test_e1)

### E2. IoU and Dice
Return `(iou, dice)` for two boolean masks. Treat two empty masks as a perfect match `(1.0, 1.0)`.

In [ ]:
def iou_dice(pred, gt):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    a = np.zeros((10, 10), bool); a[2:6, 2:6] = True     # 16 px
    b = np.zeros((10, 10), bool); b[4:8, 4:8] = True     # 16 px, overlap 4
    iou, dice = iou_dice(a, b)
    assert abs(iou - 4 / 28) < 1e-9 and abs(dice - 8 / 32) < 1e-9, f"expected (0.143, 0.25), got ({iou:.3f}, {dice:.3f})"
    assert iou_dice(a, a) == (1.0, 1.0), "identical masks should give (1, 1)"
    assert iou_dice(a, ~a)[0] == 0, "disjoint masks should give IoU 0"
    e = np.zeros((5, 5), bool)
    assert iou_dice(e, e) == (1.0, 1.0), "two empty masks should count as a perfect match"


check("E2 iou_dice", _test_e2)

### E3. A Hough accumulator
For a binary edge image, return the line accumulator `acc` of shape `(2 * diag, len(thetas))` where `diag = int(np.hypot(H, W))`: every edge pixel `(x, y)` votes for bin `round(x cos θ + y sin θ) + diag` in each column θ.

In [ ]:
def hough_accumulator(edges, thetas):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    img = np.zeros((120, 160), np.uint8)
    cv2.line(img, (10, 100), (150, 30), 255, 1)
    th = np.deg2rad(np.arange(0, 180))
    acc = np.asarray(hough_accumulator(img, th))
    diag = int(np.hypot(*img.shape))
    assert acc.shape == (2 * diag, 180), f"expected shape {(2 * diag, 180)}, got {acc.shape}"
    assert acc.sum() == (img > 0).sum() * 180, "every edge pixel must cast exactly one vote per theta"
    r, t = np.unravel_index(acc.argmax(), acc.shape)
    true_t = np.rad2deg(np.arctan2(140, 70)) % 180          # normal direction of the drawn line
    assert abs(t - true_t) <= 2 or abs(t - true_t) >= 178, f"peak at theta = {t} deg, expected about {true_t:.0f} deg"


check("E3 hough_accumulator", _test_e3)

### E4. Measure an angle (no self-check)
Use Canny + `HoughLinesP` on `building` to estimate the tilt of the building's horizontal edges in the image (median angle of long segments).
Then rotate the image to make them horizontal. What limits the accuracy of your estimate?

In [ ]:
# E4: your code here

## 7. Challenge: a lane-marking or conveyor-edge detector

Take a photo or webcam frame of a road, a corridor, or a conveyor belt with straight edges.
1. Build a pipeline: greyscale → blur → Canny → region-of-interest mask (a trapezoid in front of the camera) → `HoughLinesP`.
2. Group the segments into left and right boundaries by slope, fit one line to each, and draw them.
3. Report how the result changes with lighting and shadows. Which parameters did you have to retune?

In [ ]:
# Your challenge code here

## 8. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": "What does Otsu's method maximise?",
        "options": [
            'The number of foreground pixels',
            'The between-class variance of the two groups',
            'The image contrast',
            'The histogram peak height',
        ],
        "answer": '648015438f',
        "explain": 'Equivalently it minimises the within-class variance.',
    },
    {
        "q": 'A document photo has a shadow across one corner. Which thresholding approach is most appropriate?',
        "options": [
            'Global fixed threshold',
            'Otsu',
            'Adaptive (local) thresholding',
            'Triangle method',
        ],
        "answer": '0a730b0cc1',
        "explain": 'Each pixel gets a threshold from its own neighbourhood.',
    },
    {
        "q": 'A segmentation finds 100 defect pixels; 80 of them are real, and the ground truth has 160 defect pixels. What are precision and recall?',
        "options": [
            '0.8 and 0.5',
            '0.5 and 0.8',
            '0.8 and 0.8',
            '0.5 and 0.5',
        ],
        "answer": '3c2d45c8f5',
        "explain": 'Precision = 80/100, recall = 80/160.',
    },
    {
        "q": 'Why does Canny use two thresholds (hysteresis)?',
        "options": [
            'To detect colour edges',
            'Weak edges are kept only if connected to strong ones, which completes contours but rejects isolated noise',
            'To make edges thicker',
            'To speed up the computation',
        ],
        "answer": '0df0cc3950',
        "explain": 'Canny, 1986.',
    },
    {
        "q": 'What does non-maximum suppression do in Canny?',
        "options": [
            'Removes noise before differentiation',
            'Thins edges to one pixel by keeping only local maxima along the gradient direction',
            'Links edge segments',
            'Chooses the thresholds',
        ],
        "answer": '47d04da7e9',
        "explain": 'It is applied to the gradient magnitude.',
    },
    {
        "q": 'In the Hough transform for lines, a single edge point corresponds to what in (theta, rho) space?',
        "options": [
            'A single point',
            'A straight line',
            'A sinusoid',
            'A circle',
        ],
        "answer": '51b2b73e9f',
        "explain": 'rho = x cos(theta) + y sin(theta) for fixed (x, y).',
    },
    {
        "q": 'Why is the Hough transform robust to gaps in a line?',
        "options": [
            'It fills the gaps with interpolation',
            'Each remaining point still votes for the same accumulator bin',
            'It uses colour',
            'It needs a threshold of 0',
        ],
        "answer": '795311ad40',
        "explain": 'Missing points just mean fewer votes.',
    },
], title='Week 04A quiz')

## 9. Exit ticket (reflection)
1. Describe an inspection task where a *fixed* global threshold is actually the best choice, and what must be controlled for it to work.
2. Why is recall often more important than precision for safety-critical defect detection? Give a counter-example.
3. When would you prefer Hough circles over thresholding + contours for counting round parts?

## 10. What's new (2025–26)

> - Deep networks now produce **edge maps** (e.g. HED, and the edge-aware features inside SAM) that are far more semantic than Canny, but Canny remains
>   the default in industrial pipelines for its speed, predictability and sub-pixel accuracy (OpenCV's `EdgeDrawing` module and sub-pixel edge tools).
> - **Promptable segmentation** (SAM 2/SAM 3, Week 11) segments objects from a click or text prompt without any threshold, but classical thresholds are still used
>   to generate cheap training labels for those models (Week 9).

## 11. References and further exploration

**Seminal papers**
- Otsu, N. (1979). *A threshold selection method from gray-level histograms.* IEEE Trans. SMC 9(1). [doi:10.1109/TSMC.1979.4310076](https://doi.org/10.1109/TSMC.1979.4310076)
- Canny, J. (1986). *A computational approach to edge detection.* IEEE TPAMI 8(6). [doi:10.1109/TPAMI.1986.4767851](https://doi.org/10.1109/TPAMI.1986.4767851)
- Duda, R. O. & Hart, P. E. (1972). *Use of the Hough transformation to detect lines and curves in pictures.* Comm. ACM 15(1). [doi:10.1145/361237.361242](https://doi.org/10.1145/361237.361242)
- Sauvola, J. & Pietikäinen, M. (2000). *Adaptive document image binarization.* Pattern Recognition 33(2). [doi:10.1016/S0031-3203(99)00055-2](https://doi.org/10.1016/S0031-3203(99)00055-2)
- Zack, G. W., Rogers, W. E. & Latt, S. A. (1977). *Automatic measurement of sister chromatid exchange frequency* (the triangle threshold). J. Histochem. Cytochem. 25(7).

**Textbooks**
- Gonzalez & Woods (2018). *Digital Image Processing*, 4th ed., ch. 10 (segmentation: edges, Hough, thresholding).
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §7.2 (edges) and §7.4 (lines). [szeliski.org/Book](https://szeliski.org/Book/)
- Steger, Ulrich & Wiedemann (2018). *Machine Vision Algorithms and Applications*, 2nd ed., §3.4 (segmentation) and §3.7 (edge extraction).

**Documentation and interactive**
- OpenCV tutorials: *Image thresholding*, *Canny edge detection*, *Hough line transform*, *Hough circle transform*. [docs.opencv.org](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- scikit-image: *Thresholding* gallery (compares a dozen methods). [scikit-image.org](https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_thresholding_guide.html)
- Nayar, S. *First Principles of Computer Vision*: "Edge detection" and "Boundary detection (Hough)" lectures. [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)

**Images:** `coins`, `camera` from scikit-image (public domain); `sudoku.png`, `building.jpg` from the OpenCV sample data. Synthetic parts are generated in code.

## 12. Key takeaways
- **Thresholding** groups similar pixels: global (bimodal histograms), **Otsu** (max between-class variance), triangle and multi-Otsu, and **adaptive** for uneven light.
- **Evaluate** against ground truth with IoU/Dice and precision/recall; fix the *lighting* before blaming the algorithm.
- **Canny** = smooth → gradient → non-maximum suppression → double threshold with hysteresis.
- The **Hough transform** finds lines and circles by voting in parameter space; it tolerates gaps, noise and multiple shapes.

**Next session (04B):** from masks to objects: connected components, watershed for touching objects, contours, shape descriptors and gauging.